# CP5403 A2 – Socratic Tutor
## 01 · Preprocess course slides → FAISS index

**What this notebook does (run once):**
1. Extract text page by page from the CP5403 Week 1–7 slide PDFs
2. Clean headers/footers, page numbers and reference-only pages
3. Merge short pages into chunks with week/page metadata
4. Add a hand-written course-overview chunk (for "which week covers X?" questions)
5. Embed with `all-MiniLM-L6-v2` and save a FAISS index + `chunks.json` to Google Drive
6. Test retrieval quality

**Runtime:** CPU is enough (Runtime → Change runtime type → CPU). Keep GPU quota for fine-tuning.

**Expected Drive layout**
```
MyDrive/CP5403_A2/
├── slides/   ← 14 PDFs: week01_..._supp.pdf ... week07_..._supp.pdf
├── data/     ← socratic_train.jsonl, socratic_val.jsonl, eval_questions_v2.json
└── index/    ← created by this notebook
```

## 1. Install packages and mount Google Drive

In [ ]:
!pip install -q pymupdf sentence-transformers faiss-cpu

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Configuration and file check

In [ ]:
import os, glob

BASE      = "/content/drive/MyDrive/CP5403_A2"
SLIDE_DIR = f"{BASE}/slides"
OUT_DIR   = f"{BASE}/index"
PRACTICAL_FILE = f"{BASE}/data/practical_chunks.json"   # Week 2–7 practical guides (instructor text only)
os.makedirs(OUT_DIR, exist_ok=True)

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
MIN_WORDS   = 25     # pages shorter than this are merged into the next page
TOP_K       = 3      # chunks retrieved per question
THRESHOLD   = 0.35   # top score below this = weak / possibly out of scope (re-check in Section 8)

pdfs = sorted(glob.glob(f"{SLIDE_DIR}/*.[pP][dD][fF]"))
print("Slide folder exists:", os.path.exists(SLIDE_DIR))
print(f"Found {len(pdfs)} PDFs (expected 17)")
for p in pdfs:
    print("  ", os.path.basename(p))
print("Practical file exists:", os.path.exists(PRACTICAL_FILE))
assert pdfs, "No PDFs found – check the folder name/path, then re-run Cell 1."
assert os.path.exists(PRACTICAL_FILE), "practical_chunks.json not found in CP5403_A2/data/"

Slide folder exists: True
Found 17 PDFs (expected 17)
   week01_intro_to_llms.pdf
   week01_intro_to_llms_supp.pdf
   week02_working_with_text_data.pdf
   week02_working_with_text_data_supp.pdf
   week03_attention_mechanism.pdf
   week03_attention_mechanism_supp.pdf
   week04_implementing_gpt.pdf
   week04_implementing_gpt_supp.pdf
   week05_pretraining_unlabelled.pdf
   week05_pretraining_unlabelled_supp.pdf
   week06_finetuning_classification.pdf
   week06_finetuning_classification_supp.pdf
   week07_instruction_finetuning.pdf
   week07_instruction_finetuning_supp.pdf
   week08_rag.pdf
   week08_rag_supp.pdf
   week09_evaluating_llms.pdf
Practical file exists: True


## 3. Extract text page by page and clean it
`MuPDF error: ... Screen annotations` messages come from embedded videos and are harmless; they are switched off below.
**Expected output:** `Extracted 464 pages`

In [ ]:
import fitz, re, json
from collections import Counter
fitz.TOOLS.mupdf_display_errors(False)   # hide harmless Screen-annotation warnings

def extract_pages(pdf_path):
    with fitz.open(pdf_path) as doc:
        return [page.get_text("text") for page in doc]

def clean(text, boilerplate):
    lines = [l.strip() for l in text.splitlines()]
    lines = [l for l in lines
             if l and l not in boilerplate
             and not re.fullmatch(r"\d{1,3}", l)]          # lone page numbers
    return re.sub(r"\s+", " ", " ".join(lines)).strip()

records = []
for path in pdfs:
    fname = os.path.basename(path)
    m = re.search(r"week\s*0?(\d+)", fname, re.I)
    week = int(m.group(1)) if m else None
    pages = extract_pages(path)

    # lines that appear on more than half the pages = header/footer
    line_count = Counter(l.strip() for p in pages for l in set(p.splitlines()) if l.strip())
    boilerplate = {l for l, c in line_count.items() if c > len(pages) * 0.5}

    for i, p in enumerate(pages, start=1):
        records.append({"source": fname, "week": week,
                        "supplementary": "_supp" in fname,
                        "page": i, "text": clean(p, boilerplate)})

print(f"Extracted {len(records)} pages")
print("Empty pages (image/video only):", sum(1 for r in records if not r["text"]))
print("Files with no week number:", {r['source'] for r in records if r['week'] is None} or "none")

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Extracted 559 pages
Empty pages (image/video only): 18
Files with no week number: none


## 4. Merge short pages into chunks
- Empty pages are skipped
- Short reference-only pages (arXiv / DOI citations) are skipped
- Pages under `MIN_WORDS` are merged into the next page of the same file

In [ ]:
def is_reference_only(text):
    return bool(re.search(r"arXiv|doi\.org|\(\d{4}\)\.", text)) and len(text.split()) < 60

chunks, buffer = [], None
skipped_refs = 0
for r in records:
    if not r["text"]:
        continue
    if is_reference_only(r["text"]):
        skipped_refs += 1
        continue
    if buffer is None:
        buffer = dict(r, pages=[r["page"]])
        continue
    if buffer["source"] == r["source"] and len(buffer["text"].split()) < MIN_WORDS:
        buffer["text"] += " " + r["text"]
        buffer["pages"].append(r["page"])
        continue
    chunks.append(buffer)
    buffer = dict(r, pages=[r["page"]])
if buffer:
    chunks.append(buffer)
for c in chunks:
    c.pop("page", None)
    before = len(chunks)
    chunks = [c for c in chunks if len(c["text"].split()) >= 8]

print(f"Dropped {before - len(chunks)} tiny chunks")
print(f"Skipped reference-only pages: {skipped_refs}")
print(f"Chunks after merging: {len(chunks)}")
if chunks:
    print(chunks[min(5, len(chunks) - 1)])
else:
    print("⚠️ No chunks – check Cell 3 output")

Dropped 0 tiny chunks
Skipped reference-only pages: 19
Chunks after merging: 433
{'source': 'week01_intro_to_llms.pdf', 'week': 1, 'supplementary': False, 'text': '4. Transformer architecture Raschka, Sebastian. Build A Large Language Model (From Scratch). Manning, 2024. ISBN: 978-1633437166 Self-attention mechanisms Self-Attention Mechanism • Allows a model to focus on different parts of the input sequence when generating each word or token. • Key Equation: Attention(Q, K, V) = Softmax(QKᵀ / √d ) V ₖ • Q, K, V = Query, Key, Value vectors for each token', 'pages': [14, 15]}


## 4b. Add practical-guide chunks (Week 2–8)
Pre-processed from the practical companion guides: one chunk per section, keeping only instructor-written text (learning objectives, expected-output notes, definitions, summaries). Student answers, personal details and code blocks were removed.

In [ ]:
prac = json.load(open(PRACTICAL_FILE, encoding="utf-8"))
chunks = [c for c in chunks if c.get("type") != "practical"]   # avoid duplicates on re-run
chunks += prac
print(f"Practical chunks added: {len(prac)} (expected 134)")
print(Counter(c["week"] for c in prac))
print(prac[5]["text"][:250])

Practical chunks added: 134 (expected 134)
Counter({5: 31, 7: 26, 6: 21, 3: 19, 4: 19, 2: 18})
Practical 2 – 3. Apply tokenisation to the full text: Use the same tokenisation logic on the full text file and count how many tokens were produced. You should see the first 30 tokens from the story and then the total token count. In the original pra


## 5. Add the course-overview chunk
The training data contains logistics questions ("Which week covers attention?").
This hand-written chunk makes sure such questions can be grounded in retrieved context at inference time, matching the training format.

In [ ]:
chunks = [c for c in chunks if c["source"] != "course_overview"]   # avoid duplicates on re-run
chunks.append({
    "source": "course_overview", "week": 0, "supplementary": False, "pages": [0],
    "text": ("Course overview: Week 1 Introduction to LLMs; Week 2 Working with text data; "
             "Week 3 Coding attention mechanisms; Week 4 Implementing a GPT model; "
             "Week 5 Pretraining on unlabelled data; Week 6 Fine-tuning for classification; "
             "Week 7 Fine-tuning to follow instructions; Week 8 Retrieval-augmented generation (RAG); "
             "Week 9 Evaluating large language models.")
})
print(f"Total chunks: {len(chunks)}")
print(Counter(c["week"] for c in chunks))

Total chunks: 568
Counter({5: 88, 7: 88, 6: 78, 4: 74, 3: 73, 8: 61, 2: 54, 1: 35, 9: 16, 0: 1})


## 6. Embed and save the index to Drive
Vectors are normalised, so inner product = cosine similarity.
**Check:** `cp5403.faiss` and `chunks.json` appear in `CP5403_A2/index/`.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss, numpy as np

embedder = SentenceTransformer(EMBED_MODEL)

def chunk_text_for_embedding(c):
    prefix = "Course overview" if c["week"] == 0 else f"Week {c['week']}"
    return f"{prefix}: {c['text']}"

emb = embedder.encode([chunk_text_for_embedding(c) for c in chunks],
                      normalize_embeddings=True, show_progress_bar=True).astype("float32")

index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)

faiss.write_index(index, f"{OUT_DIR}/cp5403.faiss")
with open(f"{OUT_DIR}/chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=1)
json.dump({"embed_model": EMBED_MODEL, "n_chunks": len(chunks), "dim": int(emb.shape[1]),
           "min_words": MIN_WORDS, "threshold": THRESHOLD},
          open(f"{OUT_DIR}/index_config.json", "w"), indent=1)

print("✅ Saved:", os.listdir(OUT_DIR))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

✅ Saved: ['index_config.json', 'cp5403.faiss', 'chunks.json', 'retrieval_test_results.csv']


## 7. Retrieval functions
This is the cell to copy into the later notebooks (`03_tutor_app`, `04_evaluation`). It loads from Drive, so the PDFs are never processed again.

In [ ]:
import faiss, json
from sentence_transformers import SentenceTransformer

index    = faiss.read_index(f"{OUT_DIR}/cp5403.faiss")
chunks   = json.load(open(f"{OUT_DIR}/chunks.json", encoding="utf-8"))
embedder = SentenceTransformer(EMBED_MODEL)

def retrieve(query, k=TOP_K):
    q = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q, k)
    return [dict(chunks[i], score=float(s)) for s, i in zip(scores[0], ids[0])]

def retrieve_or_flag(query, k=TOP_K, threshold=THRESHOLD):
    """Returns (results, in_scope). in_scope=False when the best match is weak."""
    results = retrieve(query, k)
    return results, results[0]["score"] >= threshold

def format_context(results):
    """Same layout as the [Course context] block in the training data."""
    out = []
    for r in results:
        label = "Course overview" if r["week"] == 0 else f"Week {r['week']}"
        out.append(f"({label}) {r['text']}")
    return "\n".join(out)

print(f"Loaded {index.ntotal} vectors")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded 568 vectors


## 8. Retrieval test
expected is the week (or list of acceptable weeks) the answer should come from. A question passes if any of the top-3 chunks comes from an expected week. For out-of-scope questions (None), it passes if the top score is below the threshold. 📸 Screenshot this output for the Week 9 draft presentation.

In [ ]:
# (question, acceptable weeks, kind)  kind: "in" = in scope, "weak" = only briefly covered, "out" = out of scope
tests = [
    ("What is tokenisation?", [2], "in"),
    ("How does self-attention work?", [3], "in"),
    ("What is the difference between pretraining and fine-tuning?", [6], "in"),
    ("What are word embeddings?", [2], "in"),
    ("Why does GPT use a causal mask?", [3], "in"),
    ("What does layer normalisation do?", [4], "in"),
    ("How does cross-entropy loss work?", [5], "in"),
    ("What is instruction fine-tuning?", [7], "in"),
    ("Which week covers attention?", [0, 3], "in"),
    ("What is query rewriting in RAG?", [8], "in"),                          # new: Week 8
    ("What is reranking in a RAG system?", [8], "in"),                       # new: Week 8
    ("How should I compare two versions of the same LLM?", [9], "in"),       # new: Week 9
    ("Why are padding targets replaced with -100?", [7], "in"),              # new: practical guide
    ("What does temperature change during decoding?", [5], "in"),            # new: practical guide
    ("What is LoRA?", [6, 7], "weak"),
    ("What is the capital of France?", None, "out"),
]

hits, rows = 0, []
for q, expected, kind in tests:
    results, in_scope = retrieve_or_flag(q)
    weeks = [r["week"] for r in results]
    ok = (not in_scope) if kind == "out" else any(w in weeks for w in expected)
    hits += ok
    flag = "" if in_scope else "  ⚠️ below threshold"
    print(f"\n{'✅' if ok else '❌'} {q}   (expected: {expected}){flag}")
    for r in results:
        tag = "practical" if r.get("type") == "practical" else f"p{r['pages']}"
        print(f"   [Week {r['week']} {tag}] {r['score']:.2f}  {r['text'][:90]}")
    rows.append({"question": q, "expected": str(expected), "kind": kind,
                 "top_score": round(results[0]["score"], 2), "top_weeks": weeks,
                 "in_scope": in_scope, "pass": ok})

print(f"\nPassed {hits}/{len(tests)}")
in_s  = [r["top_score"] for r in rows if r["kind"] == "in"]
low_s = [r["top_score"] for r in rows if r["kind"] != "in"]
print(f"In-scope top scores: {min(in_s):.2f} – {max(in_s):.2f}")
print(f"Weak / out-of-scope top scores: {min(low_s):.2f} – {max(low_s):.2f}")
print(f"Threshold: {THRESHOLD}")

import pandas as pd
pd.DataFrame(rows).to_csv(f"{OUT_DIR}/retrieval_test_results.csv", index=False)
print("✅ Saved retrieval_test_results.csv")


✅ What is tokenisation?   (expected: [2])
   [Week 2 p[5]] 0.68  Tokens Are Pieces of Text Tokenisation means splitting text into smaller units called toke
   [Week 2 practical] 0.57  Practical 2 – Quick definitions: Token — A piece of text processed by the tokenizer. It ma
   [Week 1 p[25]] 0.57  Week 1 – Quiz Questions Question 3 Why is tokenisation needed? A. It converts human text i

✅ How does self-attention work?   (expected: [3])
   [Week 3 p[11]] 0.61  Simple Self-Attention Without Trainable Weights A simple self-attention process: 1. Compar
   [Week 3 p[33]] 0.57  Concept Discussion Discuss: • Is attention the same as human attention? • Can attention we
   [Week 1 p[12]] 0.56  Self-Attention Self-attention helps each token decide which other tokens in the input are 

✅ What is the difference between pretraining and fine-tuning?   (expected: [6])
   [Week 6 p[7]] 0.70  6.1 Categories of fine-tuning Common ways to fine tuning: Instruction finetuning Classific
   [Week 6 p[4, 5]

## 9. Notes for the report
- *Slides are pre-processed offline into page-level chunks with week and page metadata, embedded once, and persisted as a FAISS index. At inference time only the query is embedded, so retrieval latency is negligible compared with LLM generation time.*
- *A manually authored course-overview chunk was added so that logistics questions can be grounded in retrieved context.*
- *The top similarity score is used as an out-of-scope signal; topics only briefly mentioned in the slides (e.g. LoRA) receive low scores.*

**Next notebook:** `02_finetune.ipynb` (LoRA fine-tuning on `data/socratic_train.jsonl`, T4 GPU).